In [ ]:
import warnings
warnings.filterwarnings("ignore")
import torch
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
import albumentations as A
import os
import pandas as pd

# Import các module đã sửa đổi
from going_modular.dataloader.multitask import create_multitask_dataloader
from going_modular.model.MTLFaceRecognition import MTLFaceRecognition
from going_modular.loss.MultiTaskLoss import MultiTaskLoss
from going_modular.train_eval.train import fit
from going_modular.utils.transforms import RandomResizedCropRect, GaussianNoise
from going_modular.utils.MultiMetricEarlyStopping import MultiMetricEarlyStopping
from going_modular.utils.ModelCheckPoint import ModelCheckpoint

# --- CẤU HÌNH ---
device = "cuda" if torch.cuda.is_available() else "cpu"
seed = 42
torch.manual_seed(seed)


CONFIGURATION = {
    'experiment_dir': "./experiments/Single_ALBEDO_PK_SAMPLER_ConvNextV2",
    'dataset_dir': 'Photometric_DB_New',
    'type': 'albedo',

    'backbone': 'convnextv2_tiny',
    'use_sampler': True,

    'device': device,
    'epochs': 100,
    'num_workers': 4,
    'batch_size': 64,
    'image_size': 112,
    'base_lr': 1e-4,
    'num_classes': None,

    # Trọng số loss
    'loss_gender_weight': 1.0,      'loss_da_gender_weight': 0.1,
    'loss_emotion_weight': 0.5,     'loss_da_emotion_weight': 0.1,
    'loss_pose_weight': 1.0,        'loss_da_pose_weight': 0.1,
    'loss_spectacles_weight': 0.5,  'loss_da_spectacles_weight': 0.1,
    'loss_facial_hair_weight': 0.5, 'loss_da_facial_hair_weight': 0.1,
}


# --- 1. Tính num class ---
train_csv_path = os.path.join(CONFIGURATION['dataset_dir'], 'train_set.csv')
df_temp = pd.read_csv(train_csv_path)
CONFIGURATION['num_classes'] = df_temp['id'].max() + 1

# --- 2. Định nghĩa albumentation ---
train_transform = A.Compose([
    RandomResizedCropRect(CONFIGURATION['image_size']),
    GaussianNoise(p=0.2),
    A.HorizontalFlip(p=0.5)
], additional_targets={'albedo': 'image', 'depthmap': 'image', 'normal': 'image'})

test_transform = A.Compose([
    A.Resize(CONFIGURATION['image_size'], CONFIGURATION['image_size'])
], additional_targets={'albedo': 'image', 'depthmap': 'image', 'normal': 'image'})

I0000 00:00:1793169575.787374   84675 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793169575.840651   84675 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793169577.117528   84675 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
train_dataloader, test_dataloader, _ = create_multitask_dataloader(CONFIGURATION, train_transform, test_transform)
model = MTLFaceRecognition(CONFIGURATION['backbone'], CONFIGURATION['num_classes'])

criterion = MultiTaskLoss(os.path.join(CONFIGURATION['dataset_dir'], 'train_set.csv'), CONFIGURATION)
optimizer = Adam(model.parameters(), lr=CONFIGURATION['base_lr'])
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2)

early_stopping = MultiMetricEarlyStopping(
    monitor_keys=['cosine_auc'], patience=1500, mode='max', verbose=0,
    save_dir=CONFIGURATION['experiment_dir'], start_from_epoch=0
)

model_checkpoint = ModelCheckpoint(
    output_dir=CONFIGURATION['experiment_dir'],
    save_freq=5,
    verbose=1
)

fit(
    conf=CONFIGURATION,
    start_epoch=0,
    model=model,
    train_dataloader=train_dataloader,
    test_dataloader=test_dataloader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    early_stopping=early_stopping,
    model_checkpoint=model_checkpoint,
)

Epoch 1:
╒═════════════════════╤════════════╤══════════════════════╕
│ Metric              │      Train │ Test                 │
╞═════════════════════╪════════════╪══════════════════════╡
│ loss                │ 23.8218    │ -                    │
├─────────────────────┼────────────┼──────────────────────┤
│ loss_id             │ 23.411     │ -                    │
├─────────────────────┼────────────┼──────────────────────┤
│ loss_gender         │  0.0618081 │ 0.06166071444749832  │
├─────────────────────┼────────────┼──────────────────────┤
│ loss_da_gender      │  0.0581541 │ 0.06153751462697983  │
├─────────────────────┼────────────┼──────────────────────┤
│ loss_emotion        │  0.179589  │ 0.17549053728580474  │
├─────────────────────┼────────────┼──────────────────────┤
│ loss_da_emotion     │  0.181569  │ 0.17634211778640746  │
├─────────────────────┼────────────┼──────────────────────┤
│ loss_pose           │  0.0250653 │ 0.0286846786737442   │
├─────────────────────┼────────

KeyboardInterrupt: 